In [2]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)

df = data.frame

X = df.drop(columns='target')
y = df['target']

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy='median')),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

In [4]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    logistic_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

In [5]:
scores

array([0.98245614, 0.98245614, 0.97368421, 0.97368421, 0.99115044])

In [6]:
scores.mean()

np.float64(0.9806862288464524)

In [7]:
scores.std()

np.float64(0.006539441283506109)

In [8]:
print("Scores: ", scores)
print("Mean accuracy: ", scores.mean())
print("Standard deviation: ", scores.std())

Scores:  [0.98245614 0.98245614 0.97368421 0.97368421 0.99115044]
Mean accuracy:  0.9806862288464524
Standard deviation:  0.006539441283506109


In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy='median')),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)

In [10]:
rf_scores = cross_val_score(
    rf_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

In [11]:
print("RF scores: ", rf_scores)
print("RF mean accuracy: ", rf_scores.mean())
print("RF standard deviation: ", rf_scores.std())

RF scores:  [0.92105263 0.93859649 0.98245614 0.96491228 0.97345133]
RF mean accuracy:  0.9560937742586555
RF standard deviation:  0.02283883380490945


In [12]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Mean CV Accuracy": [
        scores.mean(),
        rf_scores.mean()
    ],
    "Std": [
        scores.std(),
        rf_scores.std()
    ]
})

comparison

,Model,Mean CV Accuracy,Std
0,Logistic Regression,0.980686,0.006539
1,Random Forest,0.956094,0.022839


In [13]:
f1_score = cross_val_score(
    logistic_pipeline,
    X,
    y,
    cv=5,
    scoring="f1"
)

In [14]:
print("F1 scores: ", f1_score)
print("Mean F1: ", f1_score.mean())

F1 scores:  [0.98591549 0.98611111 0.97959184 0.97931034 0.9929078 ]
Mean F1:  0.9847673174099155


In [15]:
from sklearn.model_selection import cross_validate

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

cv_results = cross_validate(
    logistic_pipeline,
    X,
    y,
    cv=5, # cross validation
    scoring=scoring
)

In [16]:
cv_results

{'fit_time': array([0.0096519 , 0.00528717, 0.00458288, 0.00428796, 0.00462484]),
 'score_time': array([0.00875688, 0.00486398, 0.00520182, 0.00455308, 0.00541496]),
 'test_accuracy': array([0.98245614, 0.98245614, 0.97368421, 0.97368421, 0.99115044]),
 'test_precision': array([0.98591549, 0.97260274, 0.96      , 0.97260274, 1.        ]),
 'test_recall': array([0.98591549, 1.        , 1.        , 0.98611111, 0.98591549]),
 'test_f1': array([0.98591549, 0.98611111, 0.97959184, 0.97931034, 0.9929078 ])}

In [17]:
pd.DataFrame(cv_results)

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1
0,0.009652,0.008757,0.982456,0.985915,0.985915,0.985915
1,0.005287,0.004864,0.982456,0.972603,1.000000,0.986111
2,0.004583,0.005202,0.973684,0.960000,1.000000,0.979592
3,0.004288,0.004553,0.973684,0.972603,0.986111,0.979310
4,0.004625,0.005415,0.991150,1.000000,0.985915,0.992908


In [18]:
rf_results = cross_validate(
    rf_pipeline,
    X,
    y,
    cv=5,
    scoring=scoring
)

pd.DataFrame(rf_results)

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1
0,0.097634,0.007012,0.921053,0.942857,0.929577,0.936170
1,0.074521,0.006591,0.938596,0.921053,0.985915,0.952381
2,0.076396,0.006952,0.982456,0.986111,0.986111,0.986111
3,0.072599,0.006508,0.964912,0.959459,0.986111,0.972603
4,0.078760,0.007189,0.973451,0.985714,0.971831,0.978723


In [20]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],

    "Fit Time": [
        cv_results["fit_time"].mean(),
        rf_results["fit_time"].mean()
    ],

        "Score Time": [
        cv_results["score_time"].mean(),
        rf_results["score_time"].mean()
    ],
    
    "Accuracy": [
        cv_results["test_accuracy"].mean(),
        rf_results["test_accuracy"].mean()
    ],
    
    "Precision": [
        cv_results["test_precision"].mean(),
        rf_results["test_precision"].mean()
    ],
    
    "Recall": [
        cv_results["test_recall"].mean(),
        rf_results["test_recall"].mean()
    ],
    
    "F1": [
        cv_results["test_f1"].mean(),
        rf_results["test_f1"].mean()
    ]
})

comparison

,Model,Fit Time,Score Time,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.005687,0.005758,0.980686,0.978224,0.991588,0.984767
1,Random Forest,0.079982,0.006850,0.956094,0.959039,0.971909,0.965198


In [21]:
print(
    "Recall:",
    cv_results["test_recall"].mean(),
    "+/-",
    cv_results["test_recall"].std()
)

Recall: 0.9915884194053209 +/- 0.006868398231182596


Cross Validate() allows us to evaluate multiple metrics across all folds, instead of relying on a single score such as accuracy.

1. What is cross validation? - Cross Validation splits data into a set number of folds and repeatedly trains and evaluates the dmoel using different folds.

2. Why is cross validation more reliable than a single train/test split? - Because model performance can vary depending on how the data is split.

3. What does cv=5 mean? - It means the data is split into 5 folds for cross validation.

4. What does the mean cross-validation score tell us? - It tells us the model's average performance across all folds.

5. What does the standard deviation of CV scores tell us? - It tells us how consistent or stable the model's performance is across the folds.

6. Why is Pipeline useful during cross validation? - Pipeline ensures that preprocessing steps are fitted only on the training folds and then applied to the validation fold, which helps prevent data leakage.

7. Do we still need a final test set after cross validation? - Yes. We still need a final test set to evaluate the selected model on completely unseen data.